# Frequency compensation
In this notebook we will demonstrate the application of frequency compensation to establish a MFM low-pass servo function.
We will use the dual-stage amplifier with EKV models from the previous notebook as starting point. 
The servo function of the uncompensated amplifier has a second-order low-pass character with two complex poles with a quality factor larger than $\frac{1}{2}\sqrt{2}$. The small-signal diagram of this amplifier is shown below.

In [22]:
import SLiCAP as sl
import sympy as sp
sl.initProject('Hearing loop', notebook=True)
sl.img2html("dualStageEKV.svg", 600)
specs     = sl.csv2specs("dualStage.csv")
spec_dict = sl.specs2dict(specs)

Compiling library: SLiCAP.lib.
Compiling library: SLiCAPmodels.lib.


## Phantom-zero compensation
Frequency compensation with phantom-zeros is by far the best method because of all methods it has:
1. a low noise penalty
2. a low power penalty
3. the lowest energy storage penalty
4. the lowest distortion penalty (phantom-zero compensation can reduce the distortion)
5. a low bandwidth penalty
## Implementation of phantom zeros
Phantom zeros can be effectively implemented as follows:
1. If elements of the feedback network(s) attenuate the loop gain at the phantom-zero frequency
  - By creating a high-pass transfer in elements of the feedback network that 
2. If the source impedance attenuates the loop gain at the frequency of the phantom zero:
   - In case of parallel feedback at the input port of the amplifier, by inserting an impedance in series with the source that establishes a low-pass transfer from the source to the amplifier
   - In case of series feedback at the input port of the amplifier, by inserting an impedance in parallel with the source that establishes a low-pass transfer from the source to the amplifier
3. If the load impedance attenuates the loop gain at the frequency of the phantom zero:
   - In case of parallel feedback at the output port of the amplifier, by inserting an impedance in series with the load that establishes a low-pass transfer from the amplifier to the load
   - In case of series feedback at the output port of the amplifier, by inserting an impedance in parallel with the load that establishes a low-pass transfer from the amplifier to the load
## Frequency of the phantom zero(s)
An MFM low-pass roll-off of a servo function of a feedback amplifier with two loop gain poles with a negative real part requires:
\begin{equation}
p_1 + p_2 = -\sqrt{2 \left( 1-L_{DC} \right) p_1 p_2}
\end{equation}
The frequency of the phantom zero:
\begin{equation}
z = -\frac{\left( 1-L_{DC} \right) p_1 p_2}{\sqrt{2 \left( 1-L_{DC} \right) p_1 p_2} + p_1 + p_2}
\end{equation}

## Frequency compensation of the dual-stage amplifier
1. An MFM low-pass characteristic of the servo function requires compensation because the quality factor of the poles exceeds $\frac{1}{2}\sqrt{2}$.
2. The required phantom zero frequency is approximately:

In [34]:
%store -r EKV_PZ_LG
L_DC  = EKV_PZ_LG.DCvalue
poles = EKV_PZ_LG.poles
zeros = EKV_PZ_LG.zeros
prodP = poles[3] * poles[4]
sumP  = poles[3] + poles[4]
z = -((1-L_DC)*prodP)/(sp.sqrt(2*(1-L_DC)*prodP) + sumP)
sl.eqn2html("z", z, units="rad/s")

3. At this frequency, the load capacitance causes a significant attenuation in the loop gain. Hence, insertion of a resistor
between the output of the amplifier and the load establishes a low-pass transfer in the asymptotic gain and an effective zero
in the loop gain. Its resistance must be:
\begin{equation}
R_{phz} = - \frac{1}{z C_L}:
\end{equation}

In [35]:
Rphz = -1/(z * spec_dict["C_L"].value)
specs.append(sl.specItem("R_phz",
                         value = Rphz,
                         description = "Phantom-zero resistance",
                         units = "Omega",
                         specType = "design"))
sl.eqn2html("R_phz", Rphz, units="Ohm")

## Verification

In [36]:
cir = sl.makeCircuit("kicad/A1/dualStageEKVcompensated.kicad_sch")
sl.specs2circuit(specs, cir)
sl.img2html("dualStageEKVcompensated.svg", 600)

Creating netlist of kicad/A1/dualStageEKVcompensated.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/A1/dualStageEKVcompensated.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/dualStageEKVcompensated.svg'.
Done.
Checking netlist: cir/dualStageEKVcompensated.cir


In [37]:
gainPZ = sl.doPZ(cir, pardefs="circuit", numeric=True)
sl.pz2html(gainPZ)

pole,Re [Hz],Im [Hz],Mag [Hz],Q
p1,$-600.6$,,$600.6$,
p2,$-106.5\cdot 10^{3}$,$106.6\cdot 10^{3}$,$150.7\cdot 10^{3}$,$0.7076$
p3,$-106.5\cdot 10^{3}$,$-106.6\cdot 10^{3}$,$150.7\cdot 10^{3}$,$0.7076$
p4,$-26.05\cdot 10^{6}$,$26.38\cdot 10^{6}$,$37.07\cdot 10^{6}$,$0.7116$
p5,$-26.05\cdot 10^{6}$,$-26.38\cdot 10^{6}$,$37.07\cdot 10^{6}$,$0.7116$
p6,$-5.662\cdot 10^{9}$,,$5.662\cdot 10^{9}$,
zero,Re [Hz],Im [Hz],Mag [Hz],Q
z1,$118.8\cdot 10^{6}$,$559.2\cdot 10^{6}$,$571.7\cdot 10^{6}$,$2.406$
z2,$118.8\cdot 10^{6}$,$-559.2\cdot 10^{6}$,$571.7\cdot 10^{6}$,$2.406$
z3,$-10.51\cdot 10^{3}$,,$10.51\cdot 10^{3}$,


In [38]:
gainLaplace = sl.doLaplace(cir, pardefs="circuit", numeric=True)
sl.plotSweep("dualStageMagComp", "Gain of compendated dual-stage amplifier", gainLaplace, 100, 1e9, 500)

In [39]:
sl.img2html("dualStageMagComp.svg", 800)

In [ ]:
sl.specs2html(specs, ["design"])

### Circuit parameters

In [40]:
sl.params2html(cir)

Name,Symbolic,Numeric
$B_{i1}$,$0.1$,$0.1$
$B_{n1}$,$0.4$,$0.4$
$CGBO_{N18}$,$1000\cdot 10^{-15}$,$1000\cdot 10^{-15}$
$CGBO_{P18}$,$1000\cdot 10^{-15}$,$1000\cdot 10^{-15}$
$CGSO_{N18}$,$300\cdot 10^{-12}$,$300\cdot 10^{-12}$
$CGSO_{P18}$,$300\cdot 10^{-12}$,$300\cdot 10^{-12}$
$CJB_{0 N18}$,$1000\cdot 10^{-6}$,$1000\cdot 10^{-6}$
$CJB_{0 P18}$,$1000\cdot 10^{-6}$,$1000\cdot 10^{-6}$
$C_{L}$,$10\cdot 10^{-12}$,$10\cdot 10^{-12}$
$C_{OX N18}$,$\frac{\epsilon_{0} \epsilon_{SiO2}}{TOX_{N18}}$,$0.008422$


**Questions**

1. Are there other effective phantom zeros possible?
2. If so, verify your answer.
3. Are there other frequency compensation techniques that establish an MFM transfer?
4. If so, verify your answer and discuss the pros and cons with respect to the one discussed above.